In [1]:
import os
import re
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import glob
import logging
logging.basicConfig(
    format='%(asctime)s [%(levelname)s]%(filename)s:%(lineno)d-%(funcName)s:%(message)s',
    level=logging.INFO  # Set the desired logging level
)
# Change directory
path = os.path.expanduser("~/scratch/data/grower/mandv/raw_data/ca/layout_investor/")

all_files = glob.glob(os.path.join(path, "*.csv"))





In [ ]:
dfs = []

for file in all_files:
    print(file)
    df = pd.read_csv(file)
    dfs.append(df)

combined_df = pd.concat(dfs, ignore_index=True)
print(combined_df.head())
combined_df.to_csv(os.path.expanduser("~/scratch/data/grower/mandv/gen_csvs/ALL_per_outage.csv"), index=False)

SyntaxError: invalid syntax (2413379783.py, line 10)

In [ ]:


combined_df['geometry'] = gpd.points_from_xy(combined_df['x'], combined_df['y'])

combined_gdf = gpd.GeoDataFrame(combined_df, geometry='geometry')
# Setting the coordinate system
combined_gdf.set_crs(epsg=4326, inplace=True)

# call zip code shape file - change directory
zip_shp= gpd.read_file(os.path.expanduser("~/Downloads/7z/v107/zip_poly.gdb"))



/home/hice1/hwang3131/scratch/data/grower/mandv/raw_data/ca/layout_investor/per_outage_investor_owned_5.csv
/home/hice1/hwang3131/scratch/data/grower/mandv/raw_data/ca/layout_investor/per_outage_investor_owned_2.csv
/home/hice1/hwang3131/scratch/data/grower/mandv/raw_data/ca/layout_investor/per_outage_investor_owned_7.csv


/tmp/ipykernel_1461155/1770993992.py:5: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


/home/hice1/hwang3131/scratch/data/grower/mandv/raw_data/ca/layout_investor/per_outage_investor_owned.csv
/home/hice1/hwang3131/scratch/data/grower/mandv/raw_data/ca/layout_investor/per_outage_investor_owned_3.csv
/home/hice1/hwang3131/scratch/data/grower/mandv/raw_data/ca/layout_investor/per_outage_investor_owned_1.csv


/tmp/ipykernel_1461155/1770993992.py:5: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


/home/hice1/hwang3131/scratch/data/grower/mandv/raw_data/ca/layout_investor/per_outage_investor_owned_8.csv


/tmp/ipykernel_1461155/1770993992.py:5: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


/home/hice1/hwang3131/scratch/data/grower/mandv/raw_data/ca/layout_investor/per_outage_investor_owned_6.csv
/home/hice1/hwang3131/scratch/data/grower/mandv/raw_data/ca/layout_investor/per_outage_investor_owned_4.csv
   OBJECTID UtilityCompany            StartDate EstimatedRestoreDate  \
0  28935115           SDGE  2023-12-19 04:35:00  2023-12-19 15:30:00   
1  28935116           SDGE  2023-12-19 06:10:00  2023-12-19 13:00:00   
2  28935117           SDGE  2023-12-19 05:59:00  2023-12-19 12:00:00   
3  28935118           SDGE  2023-12-19 04:31:00  2023-12-19 15:30:00   
4  28935119           SDGE  2023-12-19 05:12:00  2023-12-19 10:00:00   

                                               Cause  ImpactedCustomers  \
0  Upgrading the electric system in your area req...                  2   
1  Upgrading the electric system in your area req...                142   
2  Upgrading the electric system in your area req...                  1   
3  Upgrading the electric system in your area req..

/home/hice1/hwang3131/scratch/hwang3131/miniconda3/envs/grower/lib/python3.11/site-packages/pyogrio/raw.py:198: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined
  return ogr_read(


In [3]:
zip_shp.to_crs(epsg=4326, inplace=True)


In [4]:
# filter CA state
print(zip_shp[zip_shp['STATE'] == 'CA'].columns)

Index(['ZIP_CODE', 'PO_NAME', 'STATE', 'POPULATION', 'POP_SQMI', 'SQMI',
       'Shape_Length', 'Shape_Area', 'geometry'],
      dtype='object')


In [5]:
ca_shp = zip_shp[['ZIP_CODE', 'geometry']]
ca_shp = zip_shp.rename(columns = {'ZIP_CODE':'zipcode'})
print(ca_shp.columns)

Index(['zipcode', 'PO_NAME', 'STATE', 'POPULATION', 'POP_SQMI', 'SQMI',
       'Shape_Length', 'Shape_Area', 'geometry'],
      dtype='object')


In [6]:
outage_with_zip = gpd.sjoin(combined_gdf, ca_shp, how='left', predicate='within')

In [7]:
print(outage_with_zip.zipcode)
outage_with_zip.head()

0          92110
1          92105
2          92025
3          92110
4          92673
           ...  
7200230    94518
7200231    95436
7200232    93280
7200233    94122
7200234    95018
Name: zipcode, Length: 7200235, dtype: object


,OBJECTID,UtilityCompany,StartDate,EstimatedRestoreDate,Cause,ImpactedCustomers,County,OutageStatus,OutageType,GlobalID,...,geometry,index_right,zipcode,PO_NAME,STATE,POPULATION,POP_SQMI,SQMI,Shape_Length,Shape_Area
0,28935115,SDGE,2023-12-19 04:35:00,2023-12-19 15:30:00,Upgrading the electric system in your area req...,2,SAN DIEGO,Active,Planned,04974409-6b58-44a9-936a-f649a4ef2e92,...,POINT (-117.19266 32.7659),29595.0,92110,San Diego,CA,27293.0,5458.60,5.00,0.299239,0.001247
1,28935116,SDGE,2023-12-19 06:10:00,2023-12-19 13:00:00,Upgrading the electric system in your area req...,142,SAN DIEGO,Active,Planned,ed51fbf3-8439-4796-9c95-e653a21f3970,...,POINT (-117.07871 32.7456),29590.0,92105,San Diego,CA,67156.0,11802.46,5.69,0.212834,0.001418
2,28935117,SDGE,2023-12-19 05:59:00,2023-12-19 12:00:00,Upgrading the electric system in your area req...,1,SAN DIEGO,Active,Planned,a57f91ce-96c8-451f-b8b0-8f1fda0b9ab6,...,POINT (-117.07402 33.09322),29553.0,92025,Escondido,CA,52191.0,2065.33,25.27,0.580062,0.006321
3,28935118,SDGE,2023-12-19 04:31:00,2023-12-19 15:30:00,Upgrading the electric system in your area req...,2,SAN DIEGO,Active,Planned,fe223d8d-7efc-48bd-a2bd-3c62da742998,...,POINT (-117.19266 32.7659),29595.0,92110,San Diego,CA,27293.0,5458.60,5.00,0.299239,0.001247
4,28935119,SDGE,2023-12-19 05:12:00,2023-12-19 10:00:00,Upgrading the electric system in your area req...,1,ORANGE,Active,Planned,f407e0d0-77f5-4331-a9f2-da25a4fd2a93,...,POINT (-117.58904 33.45696),29816.0,92673,San Clemente,CA,30595.0,2601.62,11.76,0.295628,0.002953


In [8]:
served_df = pd.read_csv(os.path.expanduser("customers_served_county_2022.csv"))
# tolowercase
served_df["NAME"] = served_df["NAME"].str.lower()

In [9]:
from datetime import datetime, timedelta
def closest_past_15_min(dt: datetime) -> datetime:
    return dt - timedelta(minutes=dt.minute % 15, seconds=dt.second, microseconds=dt.microsecond)


In [10]:
def get_served_customers_from_county(county: str):
    county = county.lower()
    
    # Ensure no NaN values in "NAME" before str.contains()
    mask = served_df["NAME"].fillna("").str.contains(county, case=False, na=False)

    # Debug print
    # print(county, served_df[mask])
    
    return served_df.loc[mask, "customers_served"].values[0]

In [12]:
import concurrent.futures as cf
useful_cols = ["name","customersAffected","customersOutNow","customersServed","timestamp","EMC"]
def handle_per_countygroup(_provider, _county, _countygroup):
    # sort by start time
    
    _countygroup["timestamp"] = pd.to_datetime(_countygroup["timestamp"], errors='coerce')
    _countygroup = _countygroup.dropna(subset=["timestamp"])
    _countygroup.sort_values(by=['timestamp', 'ImpactedCustomers'], ascending=[True, True], inplace=True)
    
    # _countygroup.to_csv(f'{_provider}_{_county}.csv')
    firstDate = _countygroup['timestamp'].iloc[0]
    startTime = closest_past_15_min(firstDate)
    endTime = _countygroup['timestamp'].iloc[-1]
    heaper = []
    customerAffected = 0
    objectIdMap = {}
    # iterate through each row
    # (time, customerAffected, objectId) tuple
    countyDf = pd.DataFrame(columns=useful_cols)
    # find last occurrence of each objectId
    logging.info(f"{_county}|{len(_countygroup)}: Start: {startTime}, End: {endTime}")
    _countygroup['is_last'] = ~_countygroup['IncidentId'].duplicated(keep='last')
    _countygroup.to_csv(f"./csvs/{_county}_inspect.csv")
    # print(_countygroup['is_last'])
    
    subtractBuffer = 0
    lastTime = startTime
    for index, row in _countygroup.iterrows():
        customersServed = get_served_customers_from_county(_county)
        # if time is within 15 min of startTime
        while row["timestamp"] > startTime:
            startTime = startTime + timedelta(minutes=15)
            if lastTime != startTime:
                
                row_df = pd.DataFrame([{
                    "name": _county,
                    "customersAffected": customerAffected,
                    "customersOutNow": customerAffected,
                    "customersServed": customersServed,
                    "timestamp": lastTime,
                    "EMC": _provider
                }])
                lastTime = startTime
                if(customerAffected > 1e4):
                    # exit(0)
                    pass
                countyDf = pd.concat([countyDf, row_df], ignore_index=True)
                customerAffected -= subtractBuffer
                subtractBuffer = 0
                customersServed = get_served_customers_from_county(_county)
                # return countyDf
                
                # logging.info(f"ELL {row} | {customerAffected} | {subtractBuffer}")
            # heapq.heappush(heaper, (row['StartDate'], row['ImpactedCustomers'], row['OBJECTID']))
            
        # print(row_df)
        
        customerAffected += row['ImpactedCustomers']
        if row['IncidentId'] in objectIdMap:
            customerAffected -= objectIdMap[row['IncidentId']]['ImpactedCustomers']
        objectIdMap[row['IncidentId']] = row
        
        subtractBuffer += row['ImpactedCustomers'] if row['is_last'] else 0
        # logging.info(f"IFF {row} | {customerAffected} | {subtractBuffer}")
    customerAffected -= subtractBuffer
    subtractBuffer = 0
    
    # return countyDf
    row_df = pd.DataFrame([{
        "name": _county,
        "customersAffected": customerAffected,
        "customersOutNow": customerAffected,
        "customersServed": customersServed,
        "timestamp": startTime,
        "EMC": _provider
    }])
            
            
    countyDf = pd.concat([countyDf, row_df], ignore_index=True)
    countyDf["county"] = _county
    return countyDf

# oid_grouped = outage_with_zip.groupby(outage_with_zip['UtilityCompany'])
oid_grouped: list[tuple[str, pd.DataFrame]] = [("ALL_UTIL", outage_with_zip)]
with cf.ProcessPoolExecutor(16) as executor:
    for _provider, _providergroup in oid_grouped:
        # groupby county
        county_grouped = _providergroup.groupby(_providergroup['County'])
        per_util_df = pd.DataFrame(columns=useful_cols)
        county_futures = []
        cnt = 0
        for _county, _countygroup in county_grouped:
            if _county != "AMADOR":
                # continue
                pass
            # sort by start time
            county_futures.append(executor.submit(handle_per_countygroup, _provider, _county, _countygroup))
            cnt += 1
            if cnt >= 4:
                # break
                pass
        for future in cf.as_completed(county_futures):
            countyDf = future.result()
            per_util_df = pd.concat([per_util_df, countyDf], ignore_index=True)
            countyDf.sort_values(by='timestamp', inplace=True)
            assert ("county" in countyDf.columns), countyDf.columns
            
            _county = countyDf["county"].values[0]
            logging.warning(_county)
            countyDf.to_csv(f"./csvs/{_county}_v1.csv")
        # sort by timestamp
        per_util_df.sort_values(by='timestamp', inplace=True)
        per_util_df.to_csv(f'{_provider}.csv')
    

2025-03-08 13:27:24,678 [INFO]1762980492.py:21-handle_per_countygroup:ALPINE|1290: Start: 2023-04-03 20:30:00, End: 2024-08-22 23:36:09
/tmp/ipykernel_1461155/1762980492.py:47: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  countyDf = pd.concat([countyDf, row_df], ignore_index=True)
2025-03-08 13:27:24,804 [INFO]1762980492.py:21-handle_per_countygroup:AMADOR|27116: Start: 2023-03-29 22:00:00, End: 2024-11-03 18:51:09
2025-03-08 13:27:25,105 [INFO]1762980492.py:21-handle_per_countygroup:COLUSA|17432: Start: 2023-03-29 22:00:00, End: 2024-11-08 10:07:43
2025-03-08 13:27:25,110 [INFO]1762980492.py:21-handle_per_countygroup:ALAMEDA|248790: Start: 2023-03-29 22:00:00, End: 2024-12-04 07:40:03
2025-03-08 13:27:25,112 [INFO]1762980492.py:21-handle_

In [ ]:
import numpy as np
np.unique(outage_with_zip["UtilityCompany"].values)

array(['LAWP', 'PGE', 'SCE', 'SDGE', 'SMUD'], dtype=object)

In [ ]:
# all_df = pd.read_csv("ALL_UTIL.csv")
# county_dfs = all_df.groupby('name')
# for _county, _county_df in county_dfs:
#     _county_df.to_csv(f"{_county}.csv")

In [ ]:
# make the zip_shp uses the same CRS
# zip_shp.to_crs(epsg=4326, inplace=True)
# print(zip_shp)
# # filter only necessary columns
# zip_shp = zip_shp[['ZCTA5CE20', 'geometry']]
# zip_shp = zip_shp.rename(columns = {'ZCTA5CE20':'zipcode'})

# # Spatial join
# outage_with_zip = gpd.sjoin(combined_gdf, zip_shp, how='left', predicate='within')

# I think the Palo Alto has zip code information already right??